# 04 · Consumo por unidad consumidora y mercado · Resolución

Solución comentada de esta práctica de livecoding.

Seleccioná el kernel de `.venv` y usá **Reiniciar kernel y ejecutar todo**. La sección 3 contiene la función resuelta y la sección 4 ejecuta las pruebas de la práctica junto con la comprobación sobre datos reales. El resultado completo queda en la variable `resultado`.

La carga reutiliza `practicas_livecoding/preparacion.py` y los datasets ya descargados en `practicas_livecoding/datos/`. Conservá ambas carpetas dentro del mismo proyecto.

## 1. Preparación lista

La base equivale a `base` del notebook original: fecha como índice, nombres preparados y faltantes, negativos y ceros conservados.

Los archivos de `practicas_livecoding/datos/` ya están descargados. Si faltan, se descargan una vez desde Kaggle y se guardan localmente.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from pandas.testing import assert_frame_equal

# Encuentra la carpeta tanto desde la raíz del proyecto como desde este notebook.
ubicaciones = [Path.cwd(), *Path.cwd().parents]
carpeta = next(
    (
        candidata
        for ubicacion in ubicaciones
        for candidata in [ubicacion, ubicacion / "practicas_livecoding"]
        if candidata.name == "practicas_livecoding"
        and (candidata / "preparacion.py").is_file()
    ),
    None,
)
if carpeta is None:
    raise FileNotFoundError("Abrí el notebook dentro de la carpeta practicas_livecoding.")

if str(carpeta) not in sys.path:
    sys.path.insert(0, str(carpeta))

from preparacion import cargar_viviendas, cargar_energia

pd.set_option("display.max_columns", 20)

In [2]:
datos = cargar_energia()

print(f"Filas: {len(datos):,} | Columnas: {len(datos.columns)}")
display(datos.head())

Filas: 285,499 | Columnas: 13


,TipoConsumidor,Sistema,UF,Classe,Setor Econômico - N2,Setor Econômico - N3,Tipo Tensão - N1,Tipo Tensão - N2,Tipo Tensão - N3,Faixa de Consumo N1,Faixa de Consumo N2,Consumidores,Consumo
Data,,,,,,,,,,,,,
2014-01-01,Cativo,Sudeste / Centro-Oeste,RO,Residencial,Convencional (Excepto Baixa Renda),TOTAL,A - Alta Tensão,TOTAL,TOTAL,Alta Tensão,Alta Tensão,1.0,8.0
2014-01-01,Cativo,Sudeste / Centro-Oeste,SP,Rural,Serviço Público de Irrigação,TOTAL,A - Alta Tensão,"A-4 - 2,3 a 25 kV",TOTAL,Não Aplicável,Não Aplicável,6.0,145.0
2014-01-01,Cativo,Sudeste / Centro-Oeste,SP,Rural,Serviço Público de Irrigação,TOTAL,B - Baixa Tensão,TOTAL,TOTAL,Não Aplicável,Não Aplicável,31.0,18.0
2014-01-01,Cativo,Sudeste / Centro-Oeste,MS,Residencial,Baixa Renda,Baixa Renda N2,B - Baixa Tensão,TOTAL,TOTAL,Baixa Renda,> 200 kWh,44193.0,13171.0
2014-01-01,Cativo,Sistemas Isolados,AC,Residencial,Convencional (Excepto Baixa Renda),TOTAL,B - Baixa Tensão,TOTAL,TOTAL,Convencional,0-30 kWh,5216.0,149.0


## 2. Consigna y resultado esperado

Construí una tabla por mes y `TipoConsumidor` con el consumo por unidad consumidora.

1. Seleccioná primero las filas donde **ambas** mediciones, `Consumo` y `Consumidores`, estén presentes.
2. Agrupá por inicio de mes (`MS`) y mercado, y sumá ambas columnas.
3. Calculá `MWh_por_unidad = Consumo / Consumidores` solamente si el total de
   consumidores del grupo es mayor que cero; en caso contrario, dejá `NaN`.

Devolvé columnas `Consumo`, `Consumidores`, `MWh_por_unidad`, en ese orden,
y un MultiIndex llamado `Data`, `TipoConsumidor`, ordenado por fecha y mercado ascendente.
Incluí solo grupos con registros completos. Conservá consumos y consumidores negativos
al sumar; la condición del denominador se aplica **después** de agrupar.
No calcules la media de cocientes individuales ni modifiques la entrada.

## 3. Resolución comentada

1. Excluimos las filas donde falte alguna de las dos mediciones antes de agrupar.
2. `pd.Grouper(freq='MS')` combina el mes del índice con el tipo de consumidor.
3. Sumamos ambas mediciones y dividimos solo si el denominador agregado es positivo.

Es un cociente de sumas: para consumos `[10, 90]` y consumidores `[1, 3]`,
el resultado es `100 / 4 = 25`; promediar los cocientes individuales daría `20`.
Los valores negativos se conservan al sumar.

In [3]:
def calcular_intensidad_por_mercado(datos):
    """Calcula el cociente de sumas de registros completos por mes y mercado."""
    # Numerador y denominador se calculan sobre las mismas filas.
    completos = datos.dropna(subset=["Consumo", "Consumidores"])
    resultado = completos.groupby(
        [pd.Grouper(freq="MS"), "TipoConsumidor"], sort=True
    )[["Consumo", "Consumidores"]].sum(min_count=1)

    # Validamos el total de consumidores después de sumar cada grupo.
    denominador = resultado["Consumidores"].where(resultado["Consumidores"] > 0)
    resultado["MWh_por_unidad"] = resultado["Consumo"] / denominador

    return resultado

## 4. Pruebas y ejecución real

Se conservan los casos y las comprobaciones de la práctica original. Los mensajes `OK` confirman que pasan y que la entrada permanece intacta. La tabla muestra las primeras 20 filas; `resultado` contiene la salida completa.

In [4]:
# Estas pruebas no reemplazan tu función: comprueban su resultado.
def comparar(actual, esperado):
    assert isinstance(actual, pd.DataFrame), "La función debe devolver un DataFrame."
    assert_frame_equal(
        actual, esperado,
        check_dtype=False, check_index_type=False, check_column_type=False,
        check_freq=False, rtol=1e-6, atol=1e-8,
    )


def probar(funcion):
    caso = pd.DataFrame({
        "TipoConsumidor": ["Cativo", "Cativo", "Cativo", "Livre", "Livre", "Cativo", "Livre", "Livre"],
        "Consumo": [10, np.nan, 100, 20, -2, 30, -4, 5],
        "Consumidores": [2, 100, np.nan, 0, 0, 3, 2, -1],
    }, index=pd.DatetimeIndex(
        ["2023-01-01"] * 5 + ["2023-02-01"] * 2 + ["2023-03-01"], name="Data",
    ))
    original = caso.copy(deep=True)
    indice = pd.MultiIndex.from_arrays([
        pd.to_datetime(["2023-01-01", "2023-01-01", "2023-02-01", "2023-02-01", "2023-03-01"]),
        ["Cativo", "Livre", "Cativo", "Livre", "Livre"],
    ], names=["Data", "TipoConsumidor"])
    esperado = pd.DataFrame({
        "Consumo": [10, 18, 30, -4, 5],
        "Consumidores": [2, 0, 3, 2, -1],
        "MWh_por_unidad": [5, np.nan, 10, -2, np.nan],
    }, index=indice)
    comparar(funcion(caso), esperado)
    comparar(funcion(caso.iloc[::-1]), esperado)

    # Cociente de sumas: (10 + 90) / (1 + 3) = 25; la media de 10 y 30 sería 20.
    ponderado = pd.DataFrame({
        "TipoConsumidor": ["Cativo", "Cativo"],
        "Consumo": [10, 90], "Consumidores": [1, 3],
    }, index=pd.DatetimeIndex(["2023-01-01"] * 2, name="Data"))
    esperado_ponderado = esperado.iloc[:1].copy()
    esperado_ponderado.iloc[0] = [100, 4, 25]
    comparar(funcion(ponderado), esperado_ponderado)

    comparar(funcion(caso.iloc[:0]), esperado.iloc[:0])
    comparar(caso, original)
    print("OK: faltantes emparejados, cociente de sumas, negativos, denominadores y orden.")


probar(calcular_intensidad_por_mercado)

# Comprobamos también que el dataset real permanezca intacto.
antes = datos.copy(deep=True)
resultado = calcular_intensidad_por_mercado(datos)
comparar(datos, antes)
completos = datos.dropna(subset=["Consumo", "Consumidores"])
assert resultado.index.names == ["Data", "TipoConsumidor"]
assert resultado.index.is_monotonic_increasing
np.testing.assert_allclose(resultado["Consumo"].sum(), completos["Consumo"].sum())
np.testing.assert_allclose(resultado["Consumidores"].sum(), completos["Consumidores"].sum())
assert resultado.loc[resultado["Consumidores"].le(0), "MWh_por_unidad"].isna().all()
print(f"OK: dataset real. Resultado: {len(resultado):,} filas.")
display(resultado.head(20))

OK: faltantes emparejados, cociente de sumas, negativos, denominadores y orden.
OK: dataset real. Resultado: 240 filas.


Consumo  Consumidores  MWh_por_unidad
Data       TipoConsumidor                                            
2014-01-01 Cativo          2.979727e+07  7.502893e+07        0.397144
           Livre           1.012428e+07  3.532000e+03     2866.444507
2014-02-01 Cativo          3.136482e+07  7.527512e+07        0.416669
           Livre           9.987357e+06  3.551000e+03     2812.547733
2014-03-01 Cativo          2.998424e+07  7.545834e+07        0.397362
           Livre           1.000007e+07  3.581000e+03     2792.534767
2014-04-01 Cativo          2.898377e+07  7.564155e+07        0.383173
           Livre           1.020732e+07  3.592000e+03     2841.681793
2014-05-01 Cativo          2.885252e+07  7.584714e+07        0.380404
           Livre           9.923243e+06  3.614000e+03     2745.778362
2014-06-01 Cativo          2.768473e+07  7.600562e+07        0.364246
           Livre           9.717675e+06  3.640000e+03     2669.690902
2014-07-01 Cativo          2.797698e+07  7.620195e+07        0.367143
           Livre           9.544586e+06  3.657000e+03     2609.949686
2014-08-01 Cativo          2.833099e+07  7.631226e+07        0.371251
           Livre           9.885747e+06  3.667000e+03     2695.867739
2014-09-01 Cativo          2.885473e+07  7.661460e+07        0.376622
           Livre           9.707195e+06  3.663000e+03     2650.066885
2014-10-01 Cativo          3.015139e+07  7.676954e+07        0.392752
           Livre           9.626227e+06  3.669000e+03     2623.664920

## Para repasar

Volvé a la práctica sin resolver e intentá escribir la función desde cero. Explicá qué representa cada fila de la salida, cómo se resuelven los casos límite y por qué el cálculo respeta la consigna.